# Customer-Base Audit Lenses

Gold/Silver recreation of lenses 1-5. Revenue is Gold/Silver primary revenue excluding shipping.


In [1]:
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

_BOOT_ROOT = Path.cwd()
_MPL_DIR = (_BOOT_ROOT / "outputs" / ".matplotlib") if _BOOT_ROOT.name == "notebooks" else (_BOOT_ROOT / "notebooks" / "outputs" / ".matplotlib")
_MPL_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(_MPL_DIR))
os.environ.setdefault("XDG_CACHE_HOME", str(_MPL_DIR.parent / ".cache"))

import numpy as np
import pandas as pd
import matplotlib
if os.environ.get("NOTEBOOK_VALIDATION") == "1":
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

try:
    from IPython.display import display, Markdown
except Exception:
    def display(x): print(x)
    def Markdown(x): return x

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path.cwd().resolve()

SILVER = PROJECT_ROOT / "data" / "silver"
GOLD = PROJECT_ROOT / "data" / "gold"
LEGACY_OUT = PROJECT_ROOT / "EDA" / "outputs"
NB_OUT = PROJECT_ROOT / "notebooks" / "outputs"
CHART_OUT = NB_OUT / "charts"
NB_OUT.mkdir(parents=True, exist_ok=True)
CHART_OUT.mkdir(parents=True, exist_ok=True)

ANALYSIS_DATE = pd.Timestamp("2026-04-30", tz="UTC")
MARGIN = 0.40

TEAL = "#2DC4A2"
NAVY = "#1A2E44"
SLATE = "#4A6274"
ORANGE = "#F07D3E"
RED = "#E84545"
GOLD_C = "#F7B731"
LILAC = "#9B72CF"
LIGHT_BG = "#F8F9FA"
CAT_COLORS = [TEAL, NAVY, ORANGE, GOLD_C, SLATE, RED, LILAC]
plt.rcParams.update({
    "figure.facecolor": LIGHT_BG,
    "axes.facecolor": LIGHT_BG,
    "axes.edgecolor": "#E2E8ED",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": "#E2E8ED",
    "axes.labelcolor": NAVY,
    "axes.titlecolor": NAVY,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "xtick.color": SLATE,
    "ytick.color": SLATE,
    "legend.frameon": False,
    "font.family": ["DejaVu Sans"],
})

def money(x):
    return f"S${x:,.0f}"

def pct(x):
    return f"{x:.1%}"

RECREATED_CHARTS = []

def save_chart(fig, name):
    RECREATED_CHARTS.append(name)
    if os.environ.get("NOTEBOOK_VALIDATION") == "1":
        fig.canvas.draw()
    else:
        display(fig)
    plt.close(fig)
    print(f"rendered inline: {name}")
    return name

def load_marts():
    orders = pd.read_parquet(SILVER / "orders.parquet").copy()
    customers = pd.read_parquet(GOLD / "customers.parquet").copy()
    lines = pd.read_parquet(GOLD / "order_lines_enriched.parquet").copy()
    recharge_orders = pd.read_parquet(GOLD / "recharge_orders_enriched.parquet").copy()
    silver = {
        "discounts": pd.read_parquet(SILVER / "discounts.parquet"),
        "products": pd.read_parquet(SILVER / "products.parquet"),
        "campaigns": pd.read_parquet(SILVER / "campaigns.parquet"),
        "recharge_checkout": pd.read_parquet(SILVER / "recharge_checkout_items.parquet"),
        "recharge_recurring": pd.read_parquet(SILVER / "recharge_recurring_items.parquet"),
        "recharge_churned": pd.read_parquet(SILVER / "recharge_churned.parquet"),
        "recharge_reactivated": pd.read_parquet(SILVER / "recharge_reactivated.parquet"),
        "recharge_orders": pd.read_parquet(SILVER / "recharge_orders.parquet"),
    }
    orders["order_date"] = pd.to_datetime(orders["order_date"], utc=True)
    customers["first_order_date"] = pd.to_datetime(customers["first_order_date"], utc=True)
    customers["last_order_date"] = pd.to_datetime(customers["last_order_date"], utc=True)
    customers["second_order_date"] = pd.to_datetime(customers["second_order_date"], utc=True, errors="coerce")
    lines["order_date"] = pd.to_datetime(lines["order_date"], utc=True)

    orders["primary_revenue_sgd"] = pd.to_numeric(orders["order_revenue_sgd"], errors="coerce").fillna(0)
    orders["legacy_total_incl_shipping_sgd"] = pd.to_numeric(orders["order_total_incl_shipping_sgd"], errors="coerce").fillna(0)
    orders["Price: Total"] = orders["legacy_total_incl_shipping_sgd"]
    orders["Price: Total Discount"] = pd.to_numeric(orders["order_discount_sgd"], errors="coerce").fillna(0)
    orders["Price: Total Shipping"] = pd.to_numeric(orders["shipping_revenue_sgd"], errors="coerce").fillna(0)
    orders["revenue_basis_delta_sgd"] = orders["legacy_total_incl_shipping_sgd"] - orders["primary_revenue_sgd"]

    customers["total_revenue"] = pd.to_numeric(customers["total_revenue_sgd"], errors="coerce").fillna(0)
    customers["total_discount"] = pd.to_numeric(customers["total_discount_sgd"], errors="coerce").fillna(0)
    customers["first_product_cat"] = customers["first_product_category"]
    customers["is_repeat"] = customers["is_repeat"].astype(bool)

    for col in ["Line: Quantity", "Line: Price", "Line: Discount", "Line: Total"]:
        if col in lines.columns:
            lines[col] = pd.to_numeric(lines[col], errors="coerce").fillna(0)
    return orders, customers, lines, recharge_orders, silver

orders, customers, lines, recharge_orders, silver = load_marts()
print(f"Loaded marts: {len(orders):,} orders, {len(customers):,} customers, {len(lines):,} enriched lines")
print("Primary revenue excludes shipping. Legacy comparability uses order_total_incl_shipping_sgd.")


Loaded marts: 27,350 orders, 13,780 customers, 50,963 enriched lines
Primary revenue excludes shipping. Legacy comparability uses order_total_incl_shipping_sgd.


## Shared Customer-Year Tables


In [2]:
base_orders = orders.copy()
base_orders["year"] = base_orders.order_date.dt.year
cust_year = base_orders.groupby(["customer_id", "year"]).agg(n_orders=("order_id", "count"), revenue=("primary_revenue_sgd", "sum")).reset_index()
cust_year["aov"] = cust_year.revenue / cust_year.n_orders.clip(lower=1)
cust_year["profit_proxy"] = cust_year.revenue * MARGIN
first_year = base_orders.sort_values("order_date").groupby("customer_id").year.first().rename("acq_year").reset_index()
cust_year = cust_year.merge(first_year, on="customer_id", how="left")
YEARS = sorted(cust_year.year.unique())
display(cust_year.head())


,customer_id,year,n_orders,revenue,aov,profit_proxy,acq_year
0,6327387259135,2022,1,1693.12,1693.12,677.248,2022
1,6327388733695,2021,1,222.20,222.20,88.880,2021
2,6327388733695,2024,1,113.81,113.81,45.524,2021
3,6327388799231,2021,3,88.17,29.39,35.268,2021
4,6327389520127,2021,1,29.00,29.00,11.600,2021


## Lens 1: Heterogeneity


In [3]:
l1 = customers.copy()
l1["aov"] = l1.total_revenue / l1.total_orders.clip(lower=1)
l1["profit_proxy"] = l1.total_revenue * MARGIN
metrics = {"Total Transactions":"total_orders", "Total Spend":"total_revenue", "Profit Proxy":"profit_proxy", "AOV":"aov", "Lifespan Days":"lifespan_days", "Recency Days":"recency_days"}
dist = pd.DataFrame([{"metric": k, "mean": l1[v].mean(), "median": l1[v].median(), "p90": l1[v].quantile(.9), "max": l1[v].max()} for k, v in metrics.items()])
display(dist)
dist.to_csv(NB_OUT / "04_lens1_distributions.csv", index=False)

l1 = l1.sort_values("total_revenue", ascending=False).reset_index(drop=True)
l1["decile"] = pd.qcut(l1.index + 1, 10, labels=[f"D{i}" for i in range(1, 11)])
decile = l1.groupby("decile", observed=True).agg(customers=("customer_id", "count"), revenue=("total_revenue", "sum"), orders=("total_orders", "sum"), avg_spend=("total_revenue", "mean")).assign(pct_revenue=lambda d: d.revenue / d.revenue.sum(), aof=lambda d: d.orders / d.customers, aov=lambda d: d.revenue / d.orders)
display(decile)
decile.to_csv(NB_OUT / "04_lens1_decile_table.csv")
decomp = pd.DataFrame([{"customers": len(customers), "orders": len(orders), "revenue": orders.primary_revenue_sgd.sum(), "aof": len(orders) / len(customers), "aov": orders.primary_revenue_sgd.sum() / len(orders)}])
display(decomp)
decomp.to_csv(NB_OUT / "04_lens1_decomposition.csv", index=False)


,metric,mean,median,p90,max
0,Total Transactions,1.984761,1.000000,4.000000,667.000000
1,Total Spend,223.203144,65.250000,404.524000,44724.100000
2,Profit Proxy,89.281258,26.100000,161.809600,17889.640000
3,AOV,90.936778,51.212121,162.001818,34618.181818
4,Lifespan Days,117.084325,0.000000,395.000000,2256.000000
5,Recency Days,955.781640,750.000000,1912.000000,2311.000000


,customers,revenue,orders,avg_spend,pct_revenue,aof,aov
decile,,,,,,,
D1,1378,2.040875e+06,9122,1481.041258,0.663540,6.619739,223.731074
D2,1378,3.734216e+05,4045,270.988091,0.121409,2.935414,92.316833
D3,1378,2.112458e+05,2598,153.298827,0.068681,1.885341,81.310925
D4,1378,1.464604e+05,2047,106.284734,0.047618,1.485486,71.548786
D5,1378,1.040353e+05,1805,75.497281,0.033824,1.309869,57.637259
D6,1378,7.875299e+04,1663,57.150212,0.025605,1.206821,47.355979
D7,1378,5.707516e+04,1603,41.418839,0.018557,1.163280,35.605215
D8,1378,3.758041e+04,1472,27.271702,0.012218,1.068215,25.530167
D9,1378,2.304489e+04,1436,16.723432,0.007492,1.042090,16.047973


,customers,orders,revenue,aof,aov
0,13780,27350,3.075739e+06,1.984761,112.458476


## Lens 2: Period Decomposition


In [4]:
waterfall_rows = []
for a, b in zip(YEARS[:-1], YEARS[1:]):
    ca = set(cust_year[cust_year.year == a].customer_id)
    cb = set(cust_year[cust_year.year == b].customer_id)
    waterfall_rows.append({"year_a": a, "year_b": b, "n_year_a": len(ca), "n_year_b": len(cb), "n_lost": len(ca - cb), "n_retained": len(ca & cb), "n_new": len(cb - ca), "retention_rate": len(ca & cb) / len(ca) if ca else 0, "pct_new_of_year_b": len(cb - ca) / len(cb) if cb else 0})
waterfall = pd.DataFrame(waterfall_rows)
display(waterfall)
waterfall.to_csv(NB_OUT / "04_lens2_overlap_all_years.csv", index=False)

YEAR_A, YEAR_B = (2023, 2024) if {2023, 2024}.issubset(set(YEARS)) else (YEARS[-2], YEARS[-1])
a_df = cust_year[cust_year.year == YEAR_A].set_index("customer_id")
b_df = cust_year[cust_year.year == YEAR_B].set_index("customer_id")
sets = {"lost": set(a_df.index) - set(b_df.index), "retained": set(a_df.index) & set(b_df.index), "new": set(b_df.index) - set(a_df.index)}
groups = []
for label, ids in sets.items():
    source = b_df if label in ["retained", "new"] else a_df
    g = source.loc[list(ids)] if ids else source.iloc[0:0]
    groups.append({"group": label, "customers": len(ids), "orders": g.n_orders.sum(), "revenue": g.revenue.sum(), "aof": g.n_orders.sum() / len(ids) if ids else 0, "aov": g.revenue.sum() / g.n_orders.sum() if g.n_orders.sum() else 0})
group_decomp = pd.DataFrame(groups)
display(group_decomp)
group_decomp.to_csv(NB_OUT / "04_lens2_group_decomposition.csv", index=False)

both = list(sets["retained"])
if both:
    mig = pd.DataFrame({"customer_id": both, "profit_a": a_df.loc[both, "profit_proxy"], "profit_b": b_df.loc[both, "profit_proxy"]})
    mig["decile_a"] = pd.qcut(mig.profit_a.rank(method="first"), 10, labels=False) + 1
    mig["decile_b"] = pd.qcut(mig.profit_b.rank(method="first"), 10, labels=False) + 1
    migration = mig.groupby(["decile_a", "decile_b"]).size().reset_index(name="customers")
    mig["change"] = mig.profit_b - mig.profit_a
    updown = pd.DataFrame([{"segment":"up", "customers": int((mig.change > 0).sum()), "revenue_change": mig.loc[mig.change > 0, "change"].sum()}, {"segment":"down", "customers": int((mig.change < 0).sum()), "revenue_change": mig.loc[mig.change < 0, "change"].sum()}])
else:
    migration = pd.DataFrame(columns=["decile_a", "decile_b", "customers"]); updown = pd.DataFrame()
display(migration.head()); display(updown)
migration.to_csv(NB_OUT / "04_lens2_decile_migration.csv", index=False)
updown.to_csv(NB_OUT / "04_lens2_updown_analysis.csv", index=False)


,year_a,year_b,n_year_a,n_year_b,n_lost,n_retained,n_new,retention_rate,pct_new_of_year_b
0,2019,2020,3,1696,3,0,1696,0.000000,1.000000
1,2020,2021,1696,3815,1215,481,3334,0.283608,0.873919
2,2021,2022,3815,2299,3072,743,1556,0.194758,0.676816
3,2022,2023,2299,1399,1883,416,983,0.180948,0.702645
4,2023,2024,1399,2621,1042,357,2264,0.255182,0.863792
5,2024,2025,2621,4107,2152,469,3638,0.178939,0.885805
6,2025,2026,4107,1230,3788,319,911,0.077672,0.740650


,group,customers,orders,revenue,aof,aov
0,lost,1042,1355,97071.890000,1.300384,71.639771
1,retained,357,1030,84305.517273,2.885154,81.850017
2,new,2264,3231,196143.539394,1.427120,60.706759


,decile_a,decile_b,customers
0,1,1,12
1,1,2,6
2,1,3,2
3,1,4,4
4,1,5,5


,segment,customers,revenue_change
0,up,171,10343.488242
1,down,176,-9414.986788


## Lens 3: Cohort Evolution


In [5]:
FOCAL_YEAR = 2020
cohort_ids = set(first_year[first_year.acq_year == FOCAL_YEAR].customer_id)
cohort_orders = base_orders[base_orders.customer_id.isin(cohort_ids)]
cohort_size = len(cohort_ids)
annual_rows = []
for yr, g in cohort_orders.groupby("year"):
    active = g.customer_id.nunique(); rev = g.primary_revenue_sgd.sum(); n_orders = len(g)
    annual_rows.append({"year": yr, "n_active": active, "pct_active": active / cohort_size if cohort_size else 0, "total_revenue": rev, "avg_spend_per_active": rev / active if active else 0, "aof": n_orders / active if active else 0, "aov": rev / n_orders if n_orders else 0, "cohort_size": cohort_size})
cohort_annual = pd.DataFrame(annual_rows)
display(cohort_annual)
cohort_annual.to_csv(NB_OUT / "04_lens3_cohort_annual.csv", index=False)

cohort_years = sorted(cohort_orders.year.unique())
activity = cohort_orders.groupby(["customer_id", "year"]).size().unstack(fill_value=0).reindex(columns=cohort_years, fill_value=0)
patterns = activity.gt(0).replace({True:"Y", False:"N"}).agg("".join, axis=1).value_counts().reset_index()
patterns.columns = ["pattern", "customers"]
patterns["pct_cohort"] = patterns.customers / cohort_size
inter_rows = []
for cust_id, g in cohort_orders.sort_values("order_date").groupby("customer_id"):
    dates = g.order_date.tolist()
    for n in range(1, min(len(dates), 5)):
        inter_rows.append({"transition": f"{n}->{n+1}", "days": (dates[n] - dates[n-1]).days})
inter_purchase = pd.DataFrame(inter_rows)
inter_summary = inter_purchase.groupby("transition").days.describe(percentiles=[.25,.5,.75,.9]).reset_index() if len(inter_purchase) else pd.DataFrame()
display(patterns.head(10)); display(inter_summary)
patterns.to_csv(NB_OUT / "04_lens3_purchase_incidence.csv", index=False)
inter_summary.to_csv(NB_OUT / "04_lens3_inter_purchase_time.csv", index=False)

vtd = cohort_orders.groupby("customer_id").agg(vtd=("primary_revenue_sgd", "sum"), orders=("order_id", "count")).reset_index()
vtd_stats = vtd.vtd.describe(percentiles=[.25,.5,.75,.9,.95]).reset_index().rename(columns={"index":"stat", "vtd":"value"})
vtd["decile"] = pd.qcut(vtd.vtd.rank(method="first"), 10, labels=[f"D{i}" for i in range(1, 11)])
vtd_decile = vtd.groupby("decile", observed=True).agg(customers=("customer_id", "count"), revenue=("vtd", "sum"), orders=("orders", "sum")).assign(pct_vtd=lambda d: d.revenue / d.revenue.sum(), aof=lambda d: d.orders / d.customers, aov=lambda d: d.revenue / d.orders)
display(vtd_stats); display(vtd_decile)
vtd_stats.to_csv(NB_OUT / "04_lens3_vtd_distribution.csv", index=False)
vtd_decile.to_csv(NB_OUT / "04_lens3_vtd_decile_table.csv")


,year,n_active,pct_active,total_revenue,avg_spend_per_active,aof,aov,cohort_size
0,2020,1696,1.000000,449820.209697,265.224180,1.678656,157.997966,1696
1,2021,481,0.283608,293922.181515,611.064826,2.359667,258.962274,1696
2,2022,315,0.185731,200809.327879,637.489930,1.844444,345.627070,1696
3,2023,157,0.092571,35923.973636,228.815119,2.108280,108.531642,1696
4,2024,162,0.095519,37256.225455,229.976700,2.302469,99.882642,1696
5,2025,77,0.045401,16475.205758,213.963711,2.766234,77.348384,1696
6,2026,26,0.015330,3592.070000,138.156538,1.384615,99.779722,1696


,pattern,customers,pct_cohort
0,YNNNNNN,1072,0.632075
1,YYNNNNN,212,0.125000
2,YYYNNNN,96,0.056604
3,YNYNNNN,55,0.032429
4,YYYYYNN,28,0.016509
5,YYYYNNN,27,0.015920
6,YYYNYNN,22,0.012972
7,YYYYYYN,18,0.010613
8,YYNNYNN,16,0.009434
9,YNNNYNN,15,0.008844


,transition,count,mean,std,min,25%,50%,75%,90%,max
0,1->2,896.0,203.833705,310.382136,0.0,14.00,84.0,255.0,547.0,1953.0
1,2->3,603.0,186.339967,270.947486,0.0,27.50,92.0,235.5,462.8,1891.0
2,3->4,452.0,182.050885,241.890374,0.0,32.75,96.5,213.5,471.2,1944.0
3,4->5,329.0,148.762918,190.850867,0.0,28.00,86.0,173.0,398.8,919.0


,stat,value
0,count,1696.000000
1,mean,611.909902
2,std,1554.221560
3,min,0.000000
4,25%,46.000000
5,50%,124.015000
6,75%,480.418636
7,90%,1501.830000
8,95%,2666.615606
9,max,23467.800000


,customers,revenue,orders,pct_vtd,aof,aov
decile,,,,,,
D1,170,2727.235455,176,0.002628,1.035294,15.495656
D2,170,5235.886970,181,0.005045,1.064706,28.927552
D3,169,7934.822727,189,0.007646,1.118343,41.983189
D4,170,11966.836667,230,0.011531,1.352941,52.029725
D5,169,17459.276667,276,0.016823,1.633136,63.258249
D6,170,26163.846970,382,0.025211,2.247059,68.491746
D7,169,41848.226364,489,0.040324,2.893491,85.579195
D8,170,82015.270000,740,0.079028,4.352941,110.831446
D9,169,168443.776667,1069,0.162309,6.325444,157.571353


## Lens 4 and Lens 5: Vintage Quality and Base Health


In [6]:
first_purchase = base_orders.sort_values("order_date").groupby("customer_id").agg(first_date=("order_date", "min"), first_channel=("channel", "first"), first_product=("product_category", "first"), first_discount=("Price: Total Discount", "first"), first_rev=("primary_revenue_sgd", "first")).reset_index()
first_purchase["cohort_year"] = first_purchase.first_date.dt.year
first_purchase["first_disc_pct"] = (first_purchase.first_discount / first_purchase.first_rev.replace(0, np.nan)).fillna(0).clip(0, 1)
cohort_years = [y for y in sorted(first_purchase.cohort_year.unique()) if 2020 <= y <= 2025]
quality = first_purchase[first_purchase.cohort_year.isin(cohort_years)].groupby("cohort_year").agg(n_customers=("customer_id", "count"), avg_first_order_sgd=("first_rev", "mean"), pct_discounted=("first_discount", lambda s: (s > 0).mean()), avg_discount_depth=("first_disc_pct", "mean"), top_channel=("first_channel", lambda s: s.value_counts().index[0] if len(s) else None)).reset_index()
display(quality)
quality.to_csv(NB_OUT / "04_lens4_cohort_quality.csv", index=False)

orders_with_acq = base_orders.merge(first_purchase[["customer_id", "first_date", "cohort_year"]], on="customer_id", how="left")
year1_rows = []
for yr in cohort_years:
    cohort = first_purchase[first_purchase.cohort_year == yr]
    ids = set(cohort.customer_id)
    g = orders_with_acq[(orders_with_acq.customer_id.isin(ids)) & (orders_with_acq.order_date <= orders_with_acq.first_date + pd.Timedelta(days=365))]
    year1_rows.append({"cohort_year": yr, "customers": len(ids), "year1_revenue": g.primary_revenue_sgd.sum(), "year1_orders": len(g), "year1_repeat_rate": g.groupby("customer_id").size().gt(1).sum() / len(ids) if ids else 0, "year1_revenue_per_customer": g.primary_revenue_sgd.sum() / len(ids) if ids else 0})
year1 = pd.DataFrame(year1_rows)
display(year1)
year1.to_csv(NB_OUT / "04_lens4_year1_comparison.csv", index=False)

channel_mix = first_purchase[first_purchase.cohort_year.isin(cohort_years)].groupby(["cohort_year", "first_channel"]).size().reset_index(name="customers")
channel_mix["pct"] = channel_mix.groupby("cohort_year").customers.transform(lambda s: s / s.sum())
disc_intensity = first_purchase[first_purchase.cohort_year.isin(cohort_years)].groupby("cohort_year").agg(pct_discounted=("first_discount", lambda s: (s > 0).mean()), avg_discount_depth=("first_disc_pct", "mean")).reset_index()
channel_mix.to_csv(NB_OUT / "04_lens4_channel_mix_shift.csv", index=False)
disc_intensity.to_csv(NB_OUT / "04_lens4_discount_intensity.csv", index=False)

matrix = orders_with_acq.groupby(["cohort_year", "year"]).primary_revenue_sgd.sum().unstack().reset_index().rename(columns={"cohort_year":"cohort"})
display(matrix)
matrix.to_csv(NB_OUT / "04_lens5_cohort_revenue_matrix.csv", index=False)

decay_rows = []
for _, row in matrix.set_index("cohort").iterrows():
    base = row.get(row.name, np.nan)
    for yr, val in row.dropna().items():
        if yr >= row.name and pd.notna(base) and base != 0:
            decay_rows.append({"cohort": row.name, "year": int(yr), "age": int(yr - row.name), "revenue": val, "pct_of_acquisition_year": val / base})
decay = pd.DataFrame(decay_rows)
decay.to_csv(NB_OUT / "04_lens5_revenue_decay_curves.csv", index=False)

composition_rows = []
for yr in YEARS:
    active = set(cust_year[cust_year.year == yr].customer_id)
    prev = set(cust_year[cust_year.year == yr - 1].customer_id)
    acquired = set(first_year[first_year.acq_year == yr].customer_id)
    composition_rows.append({"year": yr, "new": len(active & acquired), "retained": len(active & prev), "recovered": len(active - acquired - prev), "active_customers": len(active)})
composition = pd.DataFrame(composition_rows)
display(composition)
composition.to_csv(NB_OUT / "04_lens5_composition_waterfall.csv", index=False)
scorecard = pd.DataFrame([
    {"kpi":"overall_repeat_rate", "value": customers.is_repeat.mean()},
    {"kpi":"median_days_to_second", "value": customers.loc[customers.is_repeat, "days_to_second"].median()},
    {"kpi":"pct_revenue_top_decile", "value": float(decile.iloc[0].pct_revenue)},
    {"kpi":"latest_year_new_customer_share", "value": float(composition.iloc[-1].new / composition.iloc[-1].active_customers)},
    {"kpi":"shipping_excluded_revenue_delta", "value": orders.revenue_basis_delta_sgd.sum()},
])
display(scorecard)
scorecard.to_csv(NB_OUT / "04_lens5_health_scorecard.csv", index=False)


,cohort_year,n_customers,avg_first_order_sgd,pct_discounted,avg_discount_depth,top_channel
0,2020,1696,121.544432,0.000000,0.000000,Direct / Organic
1,2021,3334,94.990555,0.000000,0.000000,Direct / Organic
2,2022,1462,128.833483,0.138851,0.055636,Direct / Organic
3,2023,873,64.508227,0.610538,0.137647,Subscription
4,2024,2015,58.035242,0.814392,0.229552,Subscription
5,2025,3524,56.986447,0.613791,0.206253,Direct / Organic


,cohort_year,customers,year1_revenue,year1_orders,year1_repeat_rate,year1_revenue_per_customer
0,2020,1696,558802.930303,3341,0.432783,329.482860
1,2021,3334,697423.948182,5771,0.346431,209.185347
2,2022,1462,348309.321515,2359,0.311902,238.241670
3,2023,873,91645.642121,1419,0.263459,104.977826
4,2024,2015,196056.692057,3263,0.286849,97.298606
5,2025,3524,355230.914138,5564,0.217934,100.803324


year,cohort,2019,2020,2021,2022,2023,2024,2025,2026
0,2019,98.127273,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020,NaN,449820.209697,293922.181515,200809.327879,35923.973636,37256.225455,16475.205758,3592.07
2,2021,NaN,NaN,544771.629697,214014.051212,29276.803939,27044.880909,10710.109394,2661.39
3,2022,NaN,NaN,NaN,326056.558788,35807.110000,26059.935152,9116.538788,1764.04
4,2023,NaN,NaN,NaN,NaN,78048.266061,22014.507879,7787.724848,1270.56
5,2024,NaN,NaN,NaN,NaN,NaN,168073.507273,45701.245693,33410.73
6,2025,NaN,NaN,NaN,NaN,NaN,NaN,314338.104138,45436.38
7,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,94477.93


,year,new,retained,recovered,active_customers
0,2019,3,0,0,3
1,2020,1696,0,0,1696
2,2021,3334,481,0,3815
3,2022,1462,743,94,2299
4,2023,873,416,110,1399
5,2024,2015,357,249,2621
6,2025,3524,469,114,4107
7,2026,873,319,38,1230


,kpi,value
0,overall_repeat_rate,0.323585
1,median_days_to_second,49.000000
2,pct_revenue_top_decile,0.663540
3,latest_year_new_customer_share,0.709756
4,shipping_excluded_revenue_delta,37212.458182
